# CDR Reconciliation in Telecom

**Purdue / Simplilearn Data Engineering Capstone**

This notebook uses PySpark to compare expected call detail record (CDR) counts with actual counts and identify mismatches.

**Input files required:** `file_count_reconciliation.txt` (pipe-delimited) and `rec_cdr_1.txt` (tab-delimited). These datasets are not included in this repository.

The file paths below reflect the original lab environment; update them for your environment before running.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when

# Create Spark Session
spark = (
    SparkSession.builder
    .appName("CDR Reconciliation")
    .getOrCreate()
)

# Read Reconciliation File
recon = (
    spark.read
    .option("header", True)
    .option("sep", "|")
    .csv("file:///home/labuser/file_count_reconciliation.txt")
)

# Read CDR File
cdr = (
    spark.read
    .option("header", True)
    .option("sep", "\t")
    .csv("file:///home/labuser/rec_cdr_1.txt")
)

# Rename reconciliation columns to match the CDR file name column
recon = (
    recon.withColumnRenamed("FileName", "CDR_FILENAME")
         .withColumnRenamed("FileCount", "Expected_Count")
)

# Count the actual records in each CDR file
actual = cdr.groupBy("CDR_FILENAME").count()
actual = actual.withColumnRenamed("count", "Actual_Count")

# Join expected counts with actual counts
result = recon.join(actual, on="CDR_FILENAME", how="left")

# Add reconciliation status
result = result.withColumn(
    "Status",
    when(
        col("Expected_Count").cast("int") == col("Actual_Count"),
        "Match"
    ).otherwise("Mismatch")
)

# Display final reconciliation results
result.show(50, False)

# Save final output to a CSV folder
(
    result.coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv("file:///home/labuser/CDR_Reconciliation_Output")
)

# Optional cloud storage examples for production use:
# result.write.mode("overwrite").option("header", True).csv(
#     "abfss://container@storageaccount.dfs.core.windows.net/CDR_Reconciliation_Output"
# )
# result.write.mode("overwrite").option("header", True).csv(
#     "s3a://bucket-name/CDR_Reconciliation_Output"
# )

print("CDR Reconciliation completed successfully.")
